# Machine Learning Data Preparation
In this notebook we scale features and extract principal components of users for unsupervised clustering in the users ML segmentation notebook

## Bootstrap
Configuring parameters, loading libraries and dataset.

In [192]:
import sys, os
# This line allows us to import from the parent directory, which is where the 'src' folder is located.
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
# These lines enable automatic reloading of modules in Jupyter, so that changes to the code are reflected without needing to restart the kernel.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Libraries
External

In [193]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler, MinMaxScaler

Project libraries

In [194]:
from src.utils import Config

### Configuration
Set here general paramenters

In [195]:
config = Config({
    'datasets_path': '../data/aggregated/',
    'save_csvs_path': '../data/processed/',
    'pca_retained_variance': 0.95,
})

### Loading Datasets

In [196]:
users = pd.read_csv(f"{config.datasets_path}/users.csv")
# Convert Users date columns to datetime format
cols_to_datetime = ['birthdate', 'sign_up_date']
users[cols_to_datetime] = users[cols_to_datetime].apply(pd.to_datetime, errors='coerce')

At this point we will remove all columns that do not represent relevant features for us:
`user_id`, `birthdate`, `home_city`, `home_airport`, `sign_up_date`, `total_sesions`

In [197]:
columns_to_remove = ['user_id', 'birthdate', 'home_city', 'home_airport', 'sign_up_date', 'total_sesions']
print("Removing features:", columns_to_remove)
users_scaled = users.drop(columns=columns_to_remove, axis=1)

Removing features: ['user_id', 'birthdate', 'home_city', 'home_airport', 'sign_up_date', 'total_sesions']


## Imputation for missing values

In [198]:
features_count_null = users_scaled.isnull().sum()
missing_values = features_count_null[features_count_null > 0]
print("Missing values in users dataset:")
for (feature, count) in missing_values.items():
  print(f"  {feature:<30} {count}")

Missing values in users dataset:
  avg_page_clicks_flight_booked  826
  avg_page_clicks_hotel_booked   597
  avg_seats_booked               1042
  avg_flight_amount              1042
  avg_flight_discount            4362
  avg_cents_km_flown             1042
  avg_hotel_amount               662
  avg_hotel_discount             4382
  avg_hotel_nights               662
  avg_checked_bags               1042
  avg_distance_flown             1042
  avg_trip_lead_time             459
  avg_trip_duration              1096
  pct_only_flight_booked         459
  pct_only_hotel_booked          459
  pct_return_flight_booked       826
  weekends_proportion            1096


Those missing values are all for users that do not have a flight or hotel booked. Therefore, we'll replace `NaN` by `0`, which is meaningful here.

In [199]:
for feature in missing_values.index:
    users_scaled.loc[users[feature].isnull(), feature] = 0

## Feature Scaling
In this section we analize the users' features and choose the best scaling method for each of them.

### Categorical Features
For categorical data, we'll map the values as the following:
* `gender`: as "Other" is a very seldom value (0.2%), we'll simply set `0` for values of `"F"` (88.2%) and `1`for other values, `["M","O"]`, (11.8%)
* `married`: we'll set `0` for `False` (56%) and `1` for `True` (44%)
* `has_children`: similarly, we'll set set `0` for `False` (67.4%) and `1` for `True` (32.6%)
* `home_counry`: has only two values, which we will attribute `0` for `"usa"` and `1` for `"canada"`

In [200]:
users_scaled['gender'] = (users_scaled['gender'] != 'F').astype(int)  # 0 for 'F', 1 for others
users_scaled['married'] = users_scaled['married'].astype(int)  # 0 for False, 1 for True
users_scaled['has_children'] = users_scaled['has_children'].astype(int)  # 0 for False, 1 for True
users_scaled['home_country'] = (users_scaled['home_country'] != 'usa').astype(int)  # 0 for 'usa', 1 for others_users = users.copy()

### Numerical Features
We can infer from the features' distribuition observed in the EAD, that the best scaler method for each would be:

| Scaler      | Application                                                      | Features            |
|-------------|------------------------------------------------------------------|---------------------|
| Standard    | Fairly normal distribuitions, not too skewed or full of outliers | age, home_airport_lat, home_airport_lon, total_trips, flight_bookings, hotel_bookings, combo_bookings, return_flight_bookings, cancellations |
| Yeo-Johnson | Havily skewed distribuitions, many outliers                      | account_age, avg_page_clicks, avg_page_clicks_flight_booked, avg_page_clicks_hotel_booked, avg_seats_booked, avg_flight_amount, total_flight_amount, avg_flight_discount, total_flight_discount, avg_hotel_amount, total_hotel_amount, avg_hotel_discount, total_hotel_discount, discount_sensitivity, avg_hotel_nights, total_hotel_nights, avg_checked_bags, total_checked_bags, avg_distance_flown, total_distance_flown |
| Robust      | Bi- or Multimodal distribuitions                                 | avg_seats_booked, avg_cents_km_flown
| Min-Max     | Categorical data and proportions (percentages)                   | gender, married, has_children, home_country, pct_only_flight_booked, pct_only_hotel_booked, weekends_proportion |

In [201]:
features_to_scale = {
    'Standard': ['age', 'home_airport_lat', 'home_airport_lon', 'total_trips', 'flight_bookings', 'hotel_bookings', 'combo_bookings', 'return_flight_bookings', 'cancellations', 'account_age', 'avg_page_clicks', 'avg_page_clicks_flight_booked', 'avg_page_clicks_hotel_booked', 'avg_seats_booked', 'avg_flight_amount', 'total_flight_amount', 'avg_flight_discount', 'total_flight_discount', 'avg_hotel_amount', 'total_hotel_amount', 'avg_hotel_discount', 'total_hotel_discount', 'discount_sensitivity', 'avg_hotel_nights', 'total_hotel_nights', 'avg_checked_bags', 'total_checked_bags', 'avg_distance_flown', 'total_distance_flown', 'avg_trip_duration', 'weekdays_travelled', 'weekends_travelled', 'avg_seats_booked', 'avg_cents_km_flown', 'avg_trip_lead_time'],
    'Min-Max': ['gender', 'married', 'has_children', 'home_country', 'pct_only_flight_booked', 'pct_only_hotel_booked', 'pct_return_flight_booked', 'weekends_proportion'],
}

In [202]:
def get_scaler(scaler_name):
  match scaler_name:
    case 'Standard': return StandardScaler()
    case 'Robust': return RobustScaler()
    case 'Yeo-Johnson': return PowerTransformer(method='yeo-johnson')
    case 'Min-Max': return MinMaxScaler()
    case _: raise ValueError(f"Scaler '{scaler_name}' not defined")

# Scaling each of the features with respective scaler
for scaler_name, features in features_to_scale.items():
  for feature in features:
    users_scaled[feature] = get_scaler(scaler_name).fit_transform(users_scaled[[feature]]).flatten()

Final dataset with all features scaled:

In [203]:
users_scaled.head(10)

,gender,married,has_children,home_country,home_airport_lat,home_airport_lon,age,account_age,total_trips,flight_bookings,...,avg_distance_flown,total_distance_flown,avg_trip_lead_time,avg_trip_duration,weekdays_travelled,weekends_travelled,pct_only_flight_booked,pct_only_hotel_booked,pct_return_flight_booked,weekends_proportion
0,0.0,1.0,0.0,0.0,0.377660,1.123663,1.946698,15.473189,-0.442626,-1.506056,...,-1.086759,-1.012767,-0.115899,-1.236835,-1.090149,-1.014899,0.0,1.0,0.0,0.000000
1,0.0,1.0,0.0,0.0,0.138518,-0.029758,0.867604,9.816766,-0.442626,-0.181522,...,-0.279934,-0.456469,-0.261837,-0.734567,-0.774163,-0.649840,0.0,0.0,1.0,0.333333
2,0.0,1.0,1.0,0.0,1.405488,-1.566140,0.784597,9.533944,-0.442626,-0.843789,...,-0.550737,-0.827976,-0.235303,0.102546,-0.774163,-0.284780,0.0,0.5,1.0,0.500000
3,0.0,1.0,0.0,0.0,0.634092,1.282326,0.120539,9.533944,1.532236,1.805280,...,-0.352570,0.252772,-0.285718,0.370422,1.121759,2.635694,0.0,0.0,1.0,0.416667
4,0.0,1.0,1.0,0.0,-2.032346,0.734891,0.286553,9.335970,-1.100914,-0.843789,...,6.343402,4.110255,4.832709,3.116152,1.753732,1.905576,0.0,0.0,1.0,0.307692
5,0.0,0.0,1.0,0.0,-0.726758,-1.341012,0.784597,8.940020,1.532236,0.480746,...,0.277017,0.397697,-0.222036,0.325776,0.015805,1.540516,0.2,0.4,1.0,0.500000
6,0.0,1.0,1.0,0.0,-0.769739,-1.311127,0.784597,8.855174,-1.100914,-0.843789,...,0.159618,-0.583085,-0.248570,2.781307,0.173798,0.445338,0.0,0.0,1.0,0.333333
7,0.0,1.0,0.0,0.0,-1.064950,-0.675368,-0.128483,8.685481,0.215661,0.480746,...,0.412833,0.538163,-0.213191,-0.120684,0.015805,0.080279,0.0,0.0,1.0,0.300000
8,0.0,0.0,0.0,0.0,-1.332370,-0.193346,-1.954642,8.459224,-1.100914,-0.843789,...,0.415705,-0.494800,-0.275104,0.102546,-0.616169,-0.649840,1.0,0.0,1.0,0.250000
9,0.0,0.0,1.0,0.0,-0.787997,0.410813,0.784597,7.950146,1.532236,1.805280,...,-0.029273,0.810048,-0.280411,0.973143,2.859686,1.905576,0.0,0.0,1.0,0.242424


## Principal Components Analysis
Here we generate a dataset based on the principal components reaching the information level defined in `config.pca_retained_variance`.

In [204]:
import warnings

# Perform PCA to retain config.pca_retained_variance of the variance
pca = PCA(n_components=config.pca_retained_variance)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    users_pca = pca.fit_transform(users_scaled)

# Create a DataFrame with the principal components
users_pca = pd.DataFrame(users_pca, columns=[f'PC{i+1}' for i in range(users_pca.shape[1])])

# Display the explained variance ratio
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.cumsum()}")
print(f"Number of components: {pca.n_components_}")

Explained variance ratio: [0.26625554 0.12021625 0.08439635 0.06392608 0.05407587 0.04653438
 0.04578174 0.0358254  0.03381528 0.02981239 0.02765338 0.02715636
 0.02416756 0.01838131 0.01715504 0.01663935 0.01302246 0.01098048
 0.00915084 0.00673462]
Cumulative explained variance: [0.26625554 0.3864718  0.47086815 0.53479423 0.5888701  0.63540448
 0.68118621 0.71701161 0.7508269  0.78063929 0.80829266 0.83544902
 0.85961659 0.8779979  0.89515294 0.91179229 0.92481475 0.93579522
 0.94494606 0.95168068]
Number of components: 20


In [205]:
users_pca.head(10)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20
0,-3.248992,-2.126463,2.977213,0.062307,0.542462,-0.671252,0.167365,1.215984,4.178629,-5.562370,13.945495,2.166883,-1.276481,0.043711,1.088867,1.299164,0.095252,0.043609,0.489728,-0.253726
1,-0.995048,0.953789,-1.086541,-1.321969,0.003929,1.245786,1.021482,-0.395234,1.264458,-4.622780,8.258523,2.372158,-0.863658,-0.399198,0.656576,-0.265682,0.272020,0.083004,0.517816,-0.039531
2,-1.675775,-0.076681,0.610355,-0.032631,-0.346866,-0.567183,0.899873,-0.357263,2.025151,-4.328439,7.777595,2.860236,-2.083423,-1.340559,0.012044,-0.567629,0.256244,-0.026875,-0.062909,0.668574
3,3.889750,-2.466920,-1.223875,-1.596559,-0.348211,-0.796586,-0.774783,1.616203,1.300741,-4.602685,8.141100,1.112124,-0.391188,-0.344820,-0.470269,0.251075,-0.259376,0.273765,0.071235,0.098989
4,16.150081,25.860228,9.611469,-7.929631,-0.861946,10.351886,4.947893,5.891551,17.563277,-3.431088,4.577778,1.969390,6.646023,-2.572595,-5.392499,1.168856,0.682279,3.294161,-0.185826,-1.238050
5,4.263649,-1.156125,2.011660,-0.439832,-0.198164,0.909277,0.915630,-1.256654,2.104479,-4.220224,7.196399,0.921324,-1.561304,-1.296823,1.126325,0.477597,-0.093327,0.242708,-0.377232,0.232884
6,-0.953901,0.452313,2.572534,-0.062936,-1.979954,-1.912882,0.217336,-1.719712,0.962182,-3.781194,7.960725,1.646646,-1.097308,-0.745076,-1.144057,0.427717,-0.770476,0.136236,0.162091,0.483389
7,1.589029,0.598637,-1.200193,-0.779495,0.875929,0.238619,0.537101,-1.452254,1.537275,-4.708456,6.908555,1.090379,-0.843642,0.212189,0.419181,0.268844,-0.071823,0.532058,0.308635,-0.014182
8,-3.146557,2.099509,-0.359186,-0.706027,-0.478528,0.196879,-1.106688,-0.941984,0.959041,-6.038190,6.342788,0.673481,0.001821,-0.167361,0.591998,-0.229784,-0.265757,-0.034634,-0.539581,-0.353503
9,5.223387,-2.865810,0.095019,-1.355098,-1.019770,-1.527782,-0.615021,0.070557,1.469105,-3.535161,6.945217,0.529439,0.038082,-0.389160,-0.404702,-0.162501,-0.196064,0.528097,-0.456620,0.336552


## Saving Datasets

In [206]:
# Keeping reference to user_id
users_scaled.insert(0, 'user_id', users['user_id'])
users_pca.insert(0, 'user_id', users['user_id'])

# Saving CSVs
users_scaled.to_csv(config.save_csvs_path + f'users_scaled.csv', index=True)
users_pca.to_csv(config.save_csvs_path + f'users_pca.csv', index=True)
print("CSVs saved successfully in " + config.save_csvs_path)

CSVs saved successfully in ../data/processed/
